[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BSLJunhyeonJeon/AI_COP/blob/main/session2/notebooks/00_setup.ipynb)

# session2 · 00 · 환경 셋업 (Setup)

- **이 노트북에서 배우는 것**: 실습 환경(코랩/로컬)이 준비됐는지 확인합니다. (개념 학습이 아니라 "준비 점검"용)
- **입력**: 없음 — 이 레포만 있으면 됩니다.
- **출력**: 실행 환경 · 프로젝트 루트(PROJECT_ROOT = session2) · GPU 사용 가능 여부 · 폴더 준비 상태

> 위에서부터 셀을 **▶ 순서대로** 누르세요. 마지막 **셀 5**의 출력만 보면 준비 완료 여부를 한눈에 알 수 있습니다.

In [ ]:
# 셀 1 · 환경 감지 + 프로젝트 루트 확보
# (이 노트북에서 '코랩 vs 로컬' 분기는 오직 이 셀 한 곳에서만 한다 — 공통 규칙: CONVENTIONS.md)
import os, subprocess

# 1) 코랩 / 로컬 감지 (표준 방식)
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

SESSION = "session2"                         # 이 노트북이 속한 세션
REPO_URL = "https://github.com/BSLJunhyeonJeon/AI_COP"
REPO_DIR = "/content/AI_COP"                 # 코랩에서 레포 클론 위치
SESSION_DIR = REPO_DIR + "/" + SESSION       # PROJECT_ROOT = 세션 폴더(레포 루트 아님)


def acquire_project():
    """코랩: 레포를 /content/AI_COP 로 클론한 뒤, 이 세션 폴더(session2)를 PROJECT_ROOT 로 반환한다.
    실패하면 None 을 반환한다(셀이 죽지 않도록).
    배포 방식을 드라이브로 바꾸려면 이 함수와 README 배지/링크만 교체하면 된다."""
    if os.path.isdir(REPO_DIR):
        print("이미 존재:", REPO_DIR, "(재클론 건너뜀)")
        try:  # 최신화 시도 — 실패해도 기존 캐시로 계속 진행
            r = subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
            if r.returncode != 0:
                print("  (git pull 실패 — 기존 캐시 버전을 사용합니다)")
        except Exception as e:
            print("  (git pull 건너뜀:", e, ")")
    else:
        print("레포 클론:", REPO_URL, "->", REPO_DIR)
        try:
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
        except Exception as e:
            print("clone 실패(네트워크/권한 확인):", e)
    # 세션 폴더가 확보됐을 때만 그 경로를 반환한다
    return SESSION_DIR if os.path.isdir(SESSION_DIR) else None


def find_root_local(marker="requirements.txt"):
    """로컬: marker(requirements.txt)를 기준으로 현재 위치에서 상위로 올라가며
    세션 루트(session2)를 찾는다. 레포 루트에는 requirements.txt 가 없어야 한다."""
    start = os.path.abspath(os.getcwd())
    d = start
    while True:
        if os.path.exists(os.path.join(d, marker)):
            return d
        parent = os.path.dirname(d)
        if parent == d:  # 최상단까지 못 찾음
            print("[주의] '" + marker + "' 를 상위 폴더에서 찾지 못했습니다. 현재 폴더를 루트로 가정합니다:", start)
            print("       (session2/ 안에서 노트북을 열었는지 확인하세요.)")
            return start
        d = parent


if IN_COLAB:
    PROJECT_ROOT = acquire_project()
else:
    PROJECT_ROOT = find_root_local()

# 루트를 확보했을 때만 chdir 한다(확보 실패 시에도 셀이 죽지 않게)
if PROJECT_ROOT and os.path.isdir(PROJECT_ROOT):
    os.chdir(PROJECT_ROOT)
    print("실행 환경   :", "Colab" if IN_COLAB else "Local")
    print("세션        :", SESSION)
    print("PROJECT_ROOT:", PROJECT_ROOT)
    print("작업 폴더   :", os.getcwd())
    print("이후 모든 경로는 이 세션 루트 기준 상대경로(data/, outputs/ ...)로 씁니다.")
else:
    print("[주의] 프로젝트(세션) 루트를 확보하지 못했습니다.")
    print("       코랩이라면 레포 클론에 실패했을 수 있습니다 — 네트워크 확인 후 이 셀을 다시 ▶ 실행하세요.")

In [ ]:
# 셀 2 · 의존성 설치
# - 0단계 baseline은 사실상 'GPU 확인용 torch' 뿐이다.
# - 코랩: torch가 사전설치되어 있으므로 재설치하지 않는다(대용량/버전 충돌 방지).
# - 로컬: torch가 없으면 requirements.txt로 설치한다.
# (셀 1을 다시 실행하지 않아도 동작하도록 여기서 다시 감지/임포트한다 — 셀 독립 실행)
import os, sys, subprocess

try:
    import torch
    print("torch 이미 설치됨:", torch.__version__, "-> 설치 단계 건너뜀")
except ImportError:
    print("torch 미설치 -> requirements.txt로 설치합니다 (로컬 환경).")
    if os.path.exists("requirements.txt"):
        r = subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], check=False)
        if r.returncode != 0:
            print("[주의] pip install 이 실패했습니다(위 로그 확인). 노트북은 계속 진행됩니다 — 셀 5로 상태를 확인하세요.")
    else:
        print("requirements.txt를 찾지 못했습니다. 셀 1을 먼저 실행하세요.")

# 참고: 1단계부터 늘어나는 추가 의존성도 requirements.txt에 적힌 뒤 이 셀에서 함께 설치됩니다.

In [ ]:
# 셀 3 · 폴더 준비 (멱등 — 있으면 그대로 두고, 없으면 만든다)
import os

# 셀 1을 다시 실행하지 않아도 올바른 위치에 만들도록, 루트가 잡혀 있으면 그 위치로 맞춘다.
# (코랩/로컬 '환경 분기'는 하지 않는다. 단지 이미 확보된 세션 루트로 cwd를 재정렬할 뿐)
root = globals().get("PROJECT_ROOT")
if root and os.path.isdir(root):
    os.chdir(root)

for d in ["data", "weights", "outputs"]:
    os.makedirs(d, exist_ok=True)
    print("준비됨:", d + "/")

In [ ]:
# 셀 4 · 데이터 다운로드 자리 (0단계는 '골격'만 — 실제 데이터는 1단계에서 채움)
def download_sample_data():
    """1단계에서 공개 출처로부터 샘플 데이터를 data/ 로 내려받는 함수가 된다.
    0단계에서는 아무것도 받지 않는다."""
    print("[자리표시] 여기서 1단계에 샘플 데이터가 data/ 로 다운로드됩니다.")
    print("0단계에서는 실제 데이터를 받지 않습니다.")


download_sample_data()

In [ ]:
# 셀 5 · 검증 (이 셀의 출력만 보면 0단계 통과 여부를 한눈에 판단)
import os, platform

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# 셀 1이 확보한 세션 루트로 맞춘 뒤 폴더 상태를 확인한다(셀 단독 재실행 대비).
root = globals().get("PROJECT_ROOT")
if root and os.path.isdir(root):
    os.chdir(root)
    root_label = root
else:
    root_label = os.getcwd() + "  (셀 1 미실행 — 현재 작업폴더 기준)"

print("=" * 52)
print(" session2 · 0단계 셋업 검증 결과")
print("=" * 52)
print("- 실행 환경      :", "Colab" if IN_COLAB else "Local")
print("- Python         :", platform.python_version())
print("- PROJECT_ROOT   :", root_label)

# GPU 사용 가능 여부 (torch 없으면 안내만 — 에러 내지 않음)
try:
    import torch
    if torch.cuda.is_available():
        print("- GPU            : 사용 가능 (" + torch.cuda.get_device_name(0) + ")")
    else:
        print("- GPU            : GPU 없음 (CPU로 동작 — 정상)")
    print("- torch 버전     :", torch.__version__, "  <- requirements.txt 핀 값으로 사용")
except ImportError:
    print("- GPU            : torch 미설치 (셀 2를 먼저 실행하세요)")

# 폴더 존재 여부
for d in ["data", "weights", "outputs"]:
    print("- 폴더 " + d.ljust(8) + " :", "있음" if os.path.isdir(d) else "없음")

print("=" * 52)
print("위 항목(환경/루트/GPU/폴더)이 모두 채워졌으면 0단계 통과입니다.")